# 深度循环神经网络

:label:`sec_deep_rnn`

到目前为止，我们只讨论了具有一个单向隐藏层的循环神经网络。
其中，隐变量和观测值与具体的函数形式的交互方式是相当随意的。
只要交互类型建模具有足够的灵活性，这就不是一个大问题。
然而，对一个单层来说，这可能具有相当的挑战性。
之前在线性模型中，我们通过添加更多的层来解决这个问题。
而在循环神经网络中，我们首先需要确定如何添加更多的层，
以及在哪里添加额外的非线性，因此这个问题有点棘手。

事实上，我们可以将多层循环神经网络堆叠在一起，
通过对几个简单层的组合，产生了一个灵活的机制。
特别是，数据可能与不同层的堆叠有关。
例如，我们可能希望保持有关金融市场状况
（熊市或牛市）的宏观数据可用，
而微观数据只记录较短期的时间动态。

 :numref:`fig_deep_rnn`描述了一个具有$L$个隐藏层的深度循环神经网络，
每个隐状态都连续地传递到当前层的下一个时间步和下一层的当前时间步。

![深度循环神经网络结构](../img/deep-rnn.svg)
:label:`fig_deep_rnn`

## 函数依赖关系

我们可以将深度架构中的函数依赖关系形式化，
这个架构是由 :numref:`fig_deep_rnn`中描述了$L$个隐藏层构成。
后续的讨论主要集中在经典的循环神经网络模型上，
但是这些讨论也适应于其他序列模型。

假设在时间步$t$有一个小批量的输入数据
$\mathbf{X}_t \in \mathbb{R}^{n \times d}$
（样本数：$n$，每个样本中的输入数：$d$）。
同时，将$l^\mathrm{th}$隐藏层（$l=1,\ldots,L$）
的隐状态设为$\mathbf{H}_t^{(l)}  \in \mathbb{R}^{n \times h}$
（隐藏单元数：$h$），
输出层变量设为$\mathbf{O}_t \in \mathbb{R}^{n \times q}$
（输出数：$q$）。
设置$\mathbf{H}_t^{(0)} = \mathbf{X}_t$，
第$l$个隐藏层的隐状态使用激活函数$\phi_l$，则：

$$\mathbf{H}_t^{(l)} = \phi_l(\mathbf{H}_t^{(l-1)} \mathbf{W}_{xh}^{(l)} + \mathbf{H}_{t-1}^{(l)} \mathbf{W}_{hh}^{(l)}  + \mathbf{b}_h^{(l)}),$$
:eqlabel:`eq_deep_rnn_H`

其中，权重$\mathbf{W}_{xh}^{(l)} \in \mathbb{R}^{h \times h}$，
$\mathbf{W}_{hh}^{(l)} \in \mathbb{R}^{h \times h}$和
偏置$\mathbf{b}_h^{(l)} \in \mathbb{R}^{1 \times h}$
都是第$l$个隐藏层的模型参数。

最后，输出层的计算仅基于第$l$个隐藏层最终的隐状态：

$$\mathbf{O}_t = \mathbf{H}_t^{(L)} \mathbf{W}_{hq} + \mathbf{b}_q,$$

其中，权重$\mathbf{W}_{hq} \in \mathbb{R}^{h \times q}$和偏置$\mathbf{b}_q \in \mathbb{R}^{1 \times q}$都是输出层的模型参数。

与多层感知机一样，隐藏层数目$L$和隐藏单元数目$h$都是超参数。
也就是说，它们可以由我们调整的。
另外，用门控循环单元或长短期记忆网络的隐状态
来代替 :eqref:`eq_deep_rnn_H`中的隐状态进行计算，
可以很容易地得到深度门控循环神经网络或深度长短期记忆神经网络。

## 简洁实现

实现多层循环神经网络所需的许多逻辑细节在高级API中都是现成的。
简单起见，我们仅示范使用此类内置函数的实现方式。
以长短期记忆网络模型为例，
该代码与之前在 :numref:`sec_lstm`中使用的代码非常相似，
实际上唯一的区别是我们指定了层的数量，
而不是使用单一层这个默认值。
像往常一样，我们从加载数据集开始。


In [2]:
import torch
from torch import nn
from d2l import torch as d2l

# 批次大小32，序列长度35
batch_size, num_steps = 32, 35

# 使用新版D2L加载时间机器数据集
data = d2l.TimeMachine(batch_size=batch_size, num_steps=num_steps)

# 获取训练数据迭代器和词表
train_iter = data.get_dataloader(train=True)
vocab = data.vocab

# 查看词表大小
print("词表大小：", len(vocab))

词表大小： 28


像选择超参数这类架构决策也跟 :numref:`sec_lstm`中的决策非常相似。
因为我们有不同的词元，所以输入和输出都选择相同数量，即`vocab_size`。
隐藏单元的数量仍然是$256$。
唯一的区别是，我们现在(**通过`num_layers`的值来设定隐藏层数**)。


In [4]:
# 词表大小、隐藏单元256、LSTM层数2
vocab_size, num_hiddens, num_layers = len(vocab), 256, 2

# 输入维度等于词表大小
num_inputs = vocab_size
device = d2l.try_gpu()

# 创建两层LSTM网络
lstm_layer = nn.LSTM(num_inputs, num_hiddens, num_layers)

In [5]:
# 封装LSTM模型
class LSTMModel(nn.Module):
    def __init__(self, lstm_layer, vocab_size):
        super().__init__()
        self.lstm = lstm_layer
        self.vocab_size = vocab_size
        self.linear = nn.Linear(num_hiddens, vocab_size)

    def forward(self, X):
        # 将字符索引转换为独热编码
        X = nn.functional.one_hot(
            X.T.long(), self.vocab_size
        ).float()

        # LSTM前向传播
        output, state = self.lstm(X)

        # 映射到词表维度
        output = self.linear(output.reshape(-1, num_hiddens))

        return output, state

# 创建模型并移动到GPU/CPU
model = LSTMModel(lstm_layer, vocab_size).to(device)

## [**训练**]与预测

由于使用了长短期记忆网络模型来实例化两个层，因此训练速度被大大降低了。


In [7]:
# 训练500轮，学习率为2
num_epochs, lr = 500, 2

# 定义损失函数和优化器
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

# 开始训练
for epoch in range(num_epochs):
    model.train()
    total_loss, total_tokens = 0.0, 0

    for X, Y in train_iter:
        # 将数据移动到GPU/CPU
        X, Y = X.to(device), Y.to(device)

        # 清空上一轮梯度
        optimizer.zero_grad()

        # 前向传播
        y_hat, _ = model(X)

        # 计算交叉熵损失
        y = Y.T.reshape(-1)
        loss = loss_fn(y_hat, y)

        # 反向传播
        loss.backward()

        # 梯度裁剪，防止梯度爆炸
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # 更新模型参数
        optimizer.step()

        # 累计损失
        total_loss += loss.item() * y.numel()
        total_tokens += y.numel()

    # 每10轮输出一次困惑度
    if (epoch + 1) % 10 == 0:
        ppl = torch.exp(
            torch.tensor(total_loss / total_tokens)
        ).item()
        print(f"epoch {epoch + 1}, perplexity {ppl:.2f}")

epoch 10, perplexity 1.34
epoch 20, perplexity 1.26
epoch 30, perplexity 1.24
epoch 40, perplexity 1.24
epoch 50, perplexity 1.23
epoch 60, perplexity 1.23
epoch 70, perplexity 1.22
epoch 80, perplexity 1.22
epoch 90, perplexity 1.22
epoch 100, perplexity 1.22
epoch 110, perplexity 1.22
epoch 120, perplexity 1.22
epoch 130, perplexity 1.22
epoch 140, perplexity 1.22
epoch 150, perplexity 1.22
epoch 160, perplexity 1.22
epoch 170, perplexity 1.22
epoch 180, perplexity 1.21
epoch 190, perplexity 1.21
epoch 200, perplexity 1.21
epoch 210, perplexity 1.21
epoch 220, perplexity 1.21
epoch 230, perplexity 1.21
epoch 240, perplexity 1.21
epoch 250, perplexity 1.21
epoch 260, perplexity 1.21
epoch 270, perplexity 1.21
epoch 280, perplexity 1.21
epoch 290, perplexity 1.21
epoch 300, perplexity 1.21
epoch 310, perplexity 1.21
epoch 320, perplexity 1.21
epoch 330, perplexity 1.21
epoch 340, perplexity 1.21
epoch 350, perplexity 1.21
epoch 360, perplexity 1.21
epoch 370, perplexity 1.21
epoch 380,

## 小结

* 在深度循环神经网络中，隐状态的信息被传递到当前层的下一时间步和下一层的当前时间步。
* 有许多不同风格的深度循环神经网络，
  如长短期记忆网络、门控循环单元、或经典循环神经网络。
  这些模型在深度学习框架的高级API中都有涵盖。
* 总体而言，深度循环神经网络需要大量的调参（如学习率和修剪）
  来确保合适的收敛，模型的初始化也需要谨慎。
